In [ ]:
import tensorflow as tf

print("TensorFlow :", tf.__version__)
print("GPU détecté :", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/DermaVision', exist_ok=True)
print("Drive monté !")

In [ ]:
# Met à jour kaggle vers la version qui supporte les clés KGA_
!pip install --upgrade kaggle -q

import json, os

# Tes valeurs depuis le fichier kaggle.json téléchargé
KAGGLE_USERNAME = "XXXXXXXXX"   # ← remplacer
KAGGLE_KEY      = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"    # ← coller clé complète

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!cat ~/.kaggle/kaggle.json
print("\nKaggle configuré !")

In [ ]:
!kaggle datasets download -d surajghuwalewala/ham1000-segmentation-and-classification --force
!unzip -q ham1000-segmentation-and-classification.zip -d ham10000
print("Dataset prêt !")
!ls ham10000

In [ ]:
import os

# Affiche toute la structure du dataset téléchargé
for root, dirs, files in os.walk('ham10000'):
    level = root.replace('ham10000', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # n'affiche les fichiers que sur les 2 premiers niveaux
        for f in files[:5]:  # limite à 5 fichiers par dossier
            print(f"  {indent}{f}")
        if len(files) > 5:
            print(f"  {indent}... ({len(files)} fichiers au total)")

In [ ]:
import pandas as pd

df = pd.read_csv('ham10000/GroundTruth.csv')

print("Colonnes :", df.columns.tolist())
print("\nAperçu :")
print(df.head())
print(f"\nNombre de lignes : {len(df)}")

In [ ]:
import os

# Label 1 = suspect (MEL + BCC + AKIEC)
# Label 0 = bénin   (NV + BKL + DF + VASC)
df['label'] = ((df['MEL'] == 1) |
               (df['BCC'] == 1) |
               (df['AKIEC'] == 1)).astype(int)

# Chemin complet vers l'image
df['path'] = df['image'].apply(
    lambda x: f'ham10000/images/{x}.jpg'
)

# Vérifie que les images existent
df = df[df['path'].apply(os.path.exists)]

print(f"Images trouvées   : {len(df)}")
print(f"Suspects (1)      : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Bénins   (0)      : {(df['label']==0).sum()} ({(1-df['label'].mean())*100:.1f}%)")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df['label_str'] = train_df['label'].astype(str)
val_df['label_str']   = val_df['label'].astype(str)

print(f"Train : {len(train_df)} images")
print(f"Val   : {len(val_df)} images")

In [ ]:
import tensorflow as tf

IMG_SIZE   = 224
BATCH_SIZE = 32

# Pas de rescale= ici — EfficientNetB4 gère son propre preprocessing
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=360,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2]
)

val_gen = tf.keras.preprocessing.image.ImageDataGenerator()  # aucune transformation

train_flow = train_gen.flow_from_dataframe(
    train_df, x_col='path', y_col='label_str',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary'
)

val_flow = val_gen.flow_from_dataframe(
    val_df, x_col='path', y_col='label_str',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print(f"Batches train : {len(train_flow)}")
print(f"Batches val   : {len(val_flow)}")

In [ ]:
import numpy as np
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense,
                                      Dropout, BatchNormalization)
from tensorflow.keras.models import Model
from sklearn.utils.class_weight import compute_class_weight

# Poids de classe — compense le déséquilibre suspects/bénins
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)
class_weights = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print(f"Poids de classe : {class_weights}")

# Architecture
base_model = EfficientNetB4(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # gelé pour l'étape 1

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)
print(f"Paramètres entraînables : {model.count_params():,}")

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=5,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/DermaVision/best_head.keras',
        monitor='val_auc', save_best_only=True, mode='max'
    )
]

print("Étape 1 — Entraînement de la tête uniquement...")
history1 = model.fit(
    train_flow, epochs=15,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks_1
)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=3, mode='max'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/DermaVision/best_finetuned.keras',
        monitor='val_auc', save_best_only=True, mode='max'
    )
]

print("Étape 2 — Fine-tuning complet...")
history2 = model.fit(
    train_flow, epochs=30,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks_2
)

In [ ]:
# On continue depuis les poids actuels (déjà fine-tunés)
# On baisse encore le LR pour des ajustements encore plus fins
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),  # 2x plus bas qu'avant
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_3 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=3, mode='max'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/DermaVision/best_final.keras',
        monitor='val_auc', save_best_only=True, mode='max'
    )
]

print("Étape 3 — Affinement final...")
history3 = model.fit(
    train_flow, epochs=15,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks_3
)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

val_flow.reset()
y_pred_proba = model.predict(val_flow, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = val_flow.classes

print("\n── Rapport de classification ──")
print(classification_report(y_true, y_pred,
      target_names=['Bénin', 'Suspect']))
print(f"AUC : {roc_auc_score(y_true, y_pred_proba):.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Bénin', 'Suspect'],
            yticklabels=['Bénin', 'Suspect'])
plt.title('Matrice de confusion')
plt.ylabel('Réel'); plt.xlabel('Prédit')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.9, 0.05)

results = []
for t in thresholds:
    y_pred_t = (y_pred_proba > t).astype(int).flatten()
    
    sensitivity  = recall_score(y_true, y_pred_t, pos_label=1)   # rappel suspects
    specificity  = recall_score(y_true, y_pred_t, pos_label=0)   # rappel bénins
    precision    = precision_score(y_true, y_pred_t, pos_label=1, zero_division=0)
    f1           = f1_score(y_true, y_pred_t, pos_label=1, zero_division=0)
    youden       = sensitivity + specificity - 1  # indice médical standard

    results.append({
        'seuil'       : round(t, 2),
        'sensibilité' : round(sensitivity, 3),   # recall suspects
        'spécificité' : round(specificity, 3),   # recall bénins
        'précision'   : round(precision, 3),
        'f1'          : round(f1, 3),
        'youden'      : round(youden, 3)
    })

import pandas as pd
df_thresh = pd.DataFrame(results)
print(df_thresh.to_string(index=False))

# ── Seuil optimal selon l'indice de Youden ────────────────────
best_youden = df_thresh.loc[df_thresh['youden'].idxmax()]
print(f"\nMeilleur seuil (Youden) : {best_youden['seuil']}")
print(f"  Sensibilité : {best_youden['sensibilité']} ({best_youden['sensibilité']*100:.1f}% des suspects détectés)")
print(f"  Spécificité : {best_youden['spécificité']} ({best_youden['spécificité']*100:.1f}% des bénins bien classés)")

# ── Seuil optimal si on veut sensibilité ≥ 0.95 ───────────────
high_sens = df_thresh[df_thresh['sensibilité'] >= 0.95]
if not high_sens.empty:
    best_sens = high_sens.loc[high_sens['spécificité'].idxmax()]
    print(f"\nMeilleur seuil (sensibilité ≥ 95%) : {best_sens['seuil']}")
    print(f"  Sensibilité : {best_sens['sensibilité']}")
    print(f"  Spécificité : {best_sens['spécificité']}")
    print(f"  Précision   : {best_sens['précision']}")

# ── Courbe ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_thresh['seuil'], df_thresh['sensibilité'], label='Sensibilité (suspects)')
ax1.plot(df_thresh['seuil'], df_thresh['spécificité'], label='Spécificité (bénins)')
ax1.plot(df_thresh['seuil'], df_thresh['précision'],   label='Précision')
ax1.axvline(x=best_youden['seuil'], color='red', linestyle='--', label=f"Youden = {best_youden['seuil']}")
ax1.set_xlabel('Seuil'); ax1.set_title('Métriques par seuil')
ax1.legend(); ax1.grid(True)

ax2.plot(df_thresh['seuil'], df_thresh['youden'], color='purple')
ax2.axvline(x=best_youden['seuil'], color='red', linestyle='--')
ax2.set_xlabel('Seuil'); ax2.set_title('Indice de Youden')
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n→ SEUIL RECOMMANDÉ POUR ANDROID : {best_youden['seuil']}")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    val_flow.reset()
    for i, (images, _) in enumerate(val_flow):
        yield [images]
        if i >= 100:
            break

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

path = '/content/drive/MyDrive/DermaVision/dermavisionv2.tflite'
with open(path, 'wb') as f:
    f.write(tflite_model)

print(f"Modèle exporté : {len(tflite_model)/1024/1024:.1f} MB")
print("Sauvegardé dans Google Drive !")

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/DermaVision/dermavision.tflite')